In [1]:
from __future__ import annotations

import os
from pathlib import Path

# ====== Paths (按需改) ======
SAM_CKPT = Path('/home/nebula/xxy/dataset/models/sam_vit_h_4b8939.pth')
YOLO_CKPT = Path('/home/nebula/xxy/dataset/models/yolov8l-world.pt')

# 默认使用 ScanNet200 mv_fast 的彩色图（数量巨大，下面会限制 N_IMAGES）
IMAGE_GLOB = '/home/nebula/xxy/dataset/data/scannet200-mv_fast/2D/*/color/*.jpg'
N_IMAGES = 12  # 先少量跑通；你可增大

# 输出目录
OUT_DIR = Path('/home/nebula/xxy/3D_Reconstruction/test/_sam_overseg_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ====== Runtime ======
DEVICE = os.environ.get('CUDA_VISIBLE_DEVICES', None)
print('OUT_DIR:', OUT_DIR)
print('SAM_CKPT exists:', SAM_CKPT.exists())
print('YOLO_CKPT exists:', YOLO_CKPT.exists())

OUT_DIR: /home/nebula/xxy/3D_Reconstruction/test/_sam_overseg_outputs
SAM_CKPT exists: True
YOLO_CKPT exists: True


In [2]:
import glob
import random

image_paths = sorted(glob.glob(IMAGE_GLOB))
if len(image_paths) == 0:
    raise FileNotFoundError(f'No images found by glob: {IMAGE_GLOB}')

# 随机抽样（便于你肉眼检查多样性）
random.seed(0)
random.shuffle(image_paths)
image_paths = image_paths[:N_IMAGES]

print('Num images:', len(image_paths))
print('Example:', image_paths[0])

Num images: 12
Example: /home/nebula/xxy/dataset/data/scannet200-mv_fast/2D/scene0028_00/color/40.jpg


In [3]:
import sys
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt

def _ensure_import(pkg: str, pip_name: str | None = None):
    try:
        __import__(pkg)
        return True
    except Exception as e:
        print(f'[WARN] import {pkg} failed: {e}')
        if pip_name is None:
            return False
        print(f'Try installing via pip: {pip_name} ...')
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])
        __import__(pkg)
        return True

# 依赖：segment_anything（SAM）
# 注意：需要联网/可访问 pip 源；若你环境已装过，会直接跳过
_ensure_import('segment_anything', pip_name='git+https://github.com/facebookresearch/segment-anything.git')

# 依赖：ultralytics（YOLO）
# 3D_Reconstruction 里可能有 FastSAM 的不完整 vendored copy；这里强制使用完整 ultralytics
_ensure_import('ultralytics', pip_name='ultralytics')

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

torch: 1.13.1
cuda available: True
device: cuda


## Part A — SAM 自动分割：统计每张图的 mask 数量

In [4]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# ====== SAM AutoMask 参数（可调） ======
# points_per_side 越大越密 -> 更容易过分割；你可以用它做对照实验
SAM_AUTO = {
    'points_per_side': 32,
    'pred_iou_thresh': 0.86,
    'stability_score_thresh': 0.92,
    'crop_n_layers': 1,
    'crop_n_points_downscale_factor': 2,
    'min_mask_region_area': 100,
}

sam = sam_model_registry['vit_h'](checkpoint=str(SAM_CKPT))
sam.to(device='cuda' if torch.cuda.is_available() else 'cpu')
mask_generator = SamAutomaticMaskGenerator(sam, **SAM_AUTO)

print('SAM auto params:', SAM_AUTO)

SAM auto params: {'points_per_side': 32, 'pred_iou_thresh': 0.86, 'stability_score_thresh': 0.92, 'crop_n_layers': 1, 'crop_n_points_downscale_factor': 2, 'min_mask_region_area': 100}


In [5]:
def read_image_bgr(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img

def bgr_to_rgb(img_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def draw_masks_overlay(rgb: np.ndarray, masks: list[dict], alpha: float = 0.55) -> np.ndarray:
    # masks: SamAutomaticMaskGenerator outputs list of dict with key 'segmentation' (H,W bool)
    out = rgb.copy()
    rng = np.random.default_rng(0)
    for m in masks:
        seg = m['segmentation']
        color = rng.integers(0, 255, size=(3,), dtype=np.uint8)
        out[seg] = (out[seg] * (1 - alpha) + color * alpha).astype(np.uint8)
    return out

def sam_auto_on_image(path: str):
    bgr = read_image_bgr(path)
    rgb = bgr_to_rgb(bgr)
    masks = mask_generator.generate(rgb)
    count = len(masks)
    areas = [int(m['area']) for m in masks]
    overlay = draw_masks_overlay(rgb, masks)
    return {
        'path': path,
        'count': count,
        'areas': areas,
        'rgb': rgb,
        'overlay': overlay,
    }

# 先跑 1 张 sanity check
demo = sam_auto_on_image(image_paths[0])
print('mask count:', demo['count'])
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1); plt.title('RGB'); plt.imshow(demo['rgb']); plt.axis('off')
plt.subplot(1, 2, 2); plt.title(f'SAM auto overlay (n={demo['count']})'); plt.imshow(demo['overlay']); plt.axis('off')
plt.show()

SyntaxError: invalid syntax (4091810997.py, line 40)

In [ ]:
import pandas as pd

rows = []
for p in image_paths:
    res = sam_auto_on_image(p)
    rows.append({
        'image': p,
        'sam_auto_mask_count': res['count'],
        'sam_auto_area_mean': float(np.mean(res['areas'])) if res['areas'] else 0.0,
        'sam_auto_area_median': float(np.median(res['areas'])) if res['areas'] else 0.0,
    })

    # 保存 overlay，方便你离线翻看
    out_path = OUT_DIR / f

    cv2.imwrite(str(out_path), cv2.cvtColor(res['overlay'], cv2.COLOR_RGB2BGR))

df_auto = pd.DataFrame(rows)
df_auto.to_csv(OUT_DIR / 'sam_auto_mask_counts.csv', index=False)
display(df_auto[['sam_auto_mask_count','sam_auto_area_mean','sam_auto_area_median','image']])

plt.figure(figsize=(6,4))
plt.hist(df_auto['sam_auto_mask_count'].values, bins=10)
plt.title('SAM auto: mask count histogram')
plt.xlabel('mask count')
plt.ylabel('images')
plt.show()

print('Saved:', OUT_DIR / 'sam_auto_mask_counts.csv')
print('Saved overlays to:', OUT_DIR)

## Part B — YOLO（框）→ SAM（box prompt）→ 选 mask → 回退（box+点）→ 去碎片/去重复\n
下面按你给的工程策略实现：\n
- YOLO：`conf=0.25~0.35`，`nms_iou=0.6`\n
- box expand：`scale=1.10`\n
- SAM：`multimask_output=True` + **Score 规则** 选 mask\n
- 失败回退：`cov<0.45 or area_norm<0.25` 时用 `1 pos(center)+2~3 neg`\n
- 清碎片：连通域过滤 / 填洞\n
- 去重复：mask-NMS（IoU>0.85 / contain>0.90）\n
- 输出监控：`avg_components_per_mask, dup_rate, fallback_rate, cov/area_norm 分布`

In [ ]:
from ultralytics import YOLO
from segment_anything import SamPredictor

# ====== YOLO 参数 ======
YOLO_CONF = 0.25
YOLO_NMS_IOU = 0.6
TOPK_PER_CLASS = 50  # 可选：防极端爆炸

# YOLO-world: 你若想做闭集，可在这里写你关心的类别；留空则用模型默认
YOLO_CLASSES: list[str] = []  # e.g. ['chair','table','sofa','bed','tv','sink']

# ====== box expand ======
BOX_EXPAND_SCALE = 1.10
MIN_WH = 24

# ====== SAM mask selection & fallback ======
FAIL_COV_THRESH = 0.45
FAIL_AREA_NORM_THRESH = 0.25
EDGE_BAND = 4  # px

# ====== postprocess ======
KEEP_TOPK_COMPONENTS = 1  # 1=仅保留最大连通域；2=保留前2
MIN_COMPONENT_AREA_ABS = 300
MIN_COMPONENT_AREA_BOX_FRAC = 0.01
HOLE_AREA_ABS = 300
HOLE_AREA_BOX_FRAC = 0.005

# ====== mask NMS ======
NMS_IOU_THRESH = 0.85
CONTAIN_THRESH = 0.90

yolo = YOLO(str(YOLO_CKPT))
if hasattr(yolo, 'set_classes') and len(YOLO_CLASSES) > 0:
    yolo.set_classes(YOLO_CLASSES)
    print('YOLO classes set:', YOLO_CLASSES)

predictor = SamPredictor(sam)

print('YOLO_CONF:', YOLO_CONF, 'YOLO_NMS_IOU:', YOLO_NMS_IOU)

AttributeError: Can't get attribute 'WorldModel' on <module 'ultralytics.nn.tasks' from '/home/nebula/xxy/ESAM/thirdparty/FastSAM/ultralytics/nn/tasks.py'>

In [ ]:
def expand_box_xyxy(box: np.ndarray, w: int, h: int, scale: float = 1.10, min_wh: int = 24) -> np.ndarray:
    x1, y1, x2, y2 = box.astype(np.float32)
    cx = 0.5 * (x1 + x2)
    cy = 0.5 * (y1 + y2)
    bw = max(float(x2 - x1), float(min_wh))
    bh = max(float(y2 - y1), float(min_wh))
    bw *= scale
    bh *= scale
    nx1 = max(0.0, cx - 0.5 * bw)
    ny1 = max(0.0, cy - 0.5 * bh)
    nx2 = min(float(w - 1), cx + 0.5 * bw)
    ny2 = min(float(h - 1), cy + 0.5 * bh)
    return np.array([nx1, ny1, nx2, ny2], dtype=np.float32)

def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = float(np.logical_and(a, b).sum())
    union = float(np.logical_or(a, b).sum())
    return 0.0 if union <= 0 else inter / union

def mask_contain(a: np.ndarray, b: np.ndarray) -> float:
    # area(intersection) / min(area(a), area(b))
    inter = float(np.logical_and(a, b).sum())
    aa = float(a.sum())
    bb = float(b.sum())
    m = min(aa, bb)
    return 0.0 if m <= 0 else inter / m

def edge_touch_ratio(mask: np.ndarray, box: np.ndarray, band: int = 4) -> float:
    x1, y1, x2, y2 = box.astype(int)
    x1 = max(0, x1); y1 = max(0, y1)
    x2 = max(x1 + 1, x2); y2 = max(y1 + 1, y2)
    m = mask[y1:y2, x1:x2]
    if m.size == 0:
        return 0.0
    area = float(m.sum())
    if area <= 0:
        return 0.0
    b = min(band, (y2 - y1) // 2, (x2 - x1) // 2)
    if b <= 0:
        return 0.0
    border = np.zeros_like(m, dtype=bool)
    border[:b, :] = True
    border[-b:, :] = True
    border[:, :b] = True
    border[:, -b:] = True
    touch = float(np.logical_and(m, border).sum())
    return touch / area

def select_best_mask(masks: np.ndarray, pred_ious: np.ndarray, box: np.ndarray) -> tuple[np.ndarray, dict]:
    # masks: (K,H,W) bool
    x1, y1, x2, y2 = box
    box_area = max(1.0, float((x2 - x1) * (y2 - y1)))

    scores = []
    infos = []
    for k in range(masks.shape[0]):
        m = masks[k]
        area = float(m.sum())
        if area <= 0:
            scores.append(-1e9)
            infos.append({'pred_iou': float(pred_ious[k]), 'cov': 0.0, 'area_norm': 0.0, 'edge_touch': 0.0, 'score': -1e9})
            continue
        bx1, by1, bx2, by2 = box.astype(int)
        bx1 = max(0, bx1); by1 = max(0, by1)
        bx2 = max(bx1 + 1, bx2); by2 = max(by1 + 1, by2)
        inter = float(m[by1:by2, bx1:bx2].sum())
        cov = inter / box_area
        area_norm = area / box_area
        et = edge_touch_ratio(m, box, band=EDGE_BAND)
        score = 0.55 * float(pred_ious[k]) + 0.25 * cov + 0.20 * area_norm - 0.30 * et
        scores.append(score)
        infos.append({'pred_iou': float(pred_ious[k]), 'cov': cov, 'area_norm': area_norm, 'edge_touch': et, 'score': score})

    best = int(np.argmax(np.asarray(scores)))
    return masks[best], infos[best]

def postprocess_mask(mask: np.ndarray, box: np.ndarray) -> tuple[np.ndarray, dict]:
    # 1) 连通域清理 2) 填洞
    m = (mask.astype(np.uint8) * 255)
    x1, y1, x2, y2 = box
    box_area = max(1.0, float((x2 - x1) * (y2 - y1)))
    min_area = int(max(MIN_COMPONENT_AREA_ABS, MIN_COMPONENT_AREA_BOX_FRAC * box_area))

    num, labels, stats, _ = cv2.connectedComponentsWithStats((m > 0).astype(np.uint8), connectivity=8)
    # stats: [label, x, y, w, h, area]
    comp_areas = []
    for i in range(1, num):
        comp_areas.append((int(stats[i, cv2.CC_STAT_AREA]), i))
    comp_areas.sort(reverse=True)

    kept = []
    for area, idx in comp_areas:
        if area < min_area:
            continue
        kept.append(idx)
        if len(kept) >= KEEP_TOPK_COMPONENTS:
            break

    cleaned = np.zeros_like(m, dtype=np.uint8)
    for idx in kept:
        cleaned[labels == idx] = 255

    # 填洞（flood fill）
    hole_area = int(max(HOLE_AREA_ABS, HOLE_AREA_BOX_FRAC * box_area))
    inv = (cleaned == 0).astype(np.uint8) * 255
    ff = inv.copy()
    h, w = ff.shape
    mask_ff = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(ff, mask_ff, (0, 0), 0)
    holes = (ff > 0).astype(np.uint8)
    # 只填小洞：按连通域过滤 holes
    numh, labelh, statsh, _ = cv2.connectedComponentsWithStats(holes, connectivity=8)
    fill = np.zeros_like(holes)
    for i in range(1, numh):
        a = int(statsh[i, cv2.CC_STAT_AREA])
        if a <= hole_area:
            fill[labelh == i] = 1
    filled = cleaned.copy()
    filled[fill > 0] = 255

    out = (filled > 0)
    info = {
        'components_before': max(0, num - 1),
        'components_kept': len(kept),
        'min_component_area': min_area,
    }
    return out, info

def mask_nms(instances: list[dict]) -> tuple[list[dict], int]:
    # instances: [{'mask': bool(H,W), 'cls': int, 'det_conf': float, 'mask_score': float}]
    # 返回保留实例 + 被抑制数
    if len(instances) == 0:
        return [], 0

    # 按 combined score 排序
    order = sorted(range(len(instances)), key=lambda i: instances[i]['det_conf'] * instances[i]['mask_score'], reverse=True)
    keep = []
    suppressed = 0
    for idx in order:
        cand = instances[idx]
        ok = True
        for j in keep:
            ref = instances[j]
            # 同类优先比较（也可改成全类）
            if cand['cls'] != ref['cls']:
                continue
            iou = mask_iou(cand['mask'], ref['mask'])
            contain = mask_contain(cand['mask'], ref['mask'])
            if iou > NMS_IOU_THRESH or contain > CONTAIN_THRESH:
                ok = False
                suppressed += 1
                break
        if ok:
            keep.append(idx)
    kept = [instances[i] for i in keep]
    return kept, suppressed

In [ ]:
def yolo_predict(rgb: np.ndarray):
    # ultralytics 接收 numpy RGB/BGR 都能跑，但这里统一传 RGB
    res = yolo.predict(rgb, conf=YOLO_CONF, iou=YOLO_NMS_IOU, verbose=False)[0]
    boxes = res.boxes
    if boxes is None or len(boxes) == 0:
        return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.float32), np.zeros((0,), dtype=np.int64), getattr(res, 'names', {})
    xyxy = boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    conf = boxes.conf.detach().cpu().numpy().astype(np.float32)
    cls = boxes.cls.detach().cpu().numpy().astype(np.int64)
    return xyxy, conf, cls, getattr(res, 'names', {})

def sample_negative_points(box: np.ndarray, coarse_mask: np.ndarray, k: int = 3) -> np.ndarray:
    x1, y1, x2, y2 = box.astype(int)
    x1 = max(0, x1); y1 = max(0, y1)
    x2 = max(x1 + 2, x2); y2 = max(y1 + 2, y2)

    candidates = [
        (x1 + 5, y1 + 5),
        (x2 - 5, y1 + 5),
        (x1 + 5, y2 - 5),
        (x2 - 5, y2 - 5),
    ]

    negs = []
    for (x, y) in candidates:
        x = int(np.clip(x, x1, x2 - 1))
        y = int(np.clip(y, y1, y2 - 1))
        if not coarse_mask[y, x]:
            negs.append([x, y])
        if len(negs) >= k:
            break

    # 兜底：随机采样 box 内不在 coarse_mask 的点
    if len(negs) < k:
        ys, xs = np.where(~coarse_mask[y1:y2, x1:x2])
        if len(xs) > 0:
            idx = np.random.choice(len(xs), size=min(k - len(negs), len(xs)), replace=False)
            for ii in idx:
                negs.append([int(x1 + xs[ii]), int(y1 + ys[ii])])

    return np.array(negs, dtype=np.float32) if len(negs) > 0 else np.zeros((0, 2), dtype=np.float32)

def sam_segment_from_box(rgb: np.ndarray, boxes_xyxy: np.ndarray, confs: np.ndarray, clses: np.ndarray):
    H, W = rgb.shape[:2]
    predictor.set_image(rgb)

    instances = []
    fallback_used = 0
    comps_total = 0

    for box, det_conf, cls_id in zip(boxes_xyxy, confs, clses):
        ebox = expand_box_xyxy(box, W, H, scale=BOX_EXPAND_SCALE, min_wh=MIN_WH)

        masks, scores, _ = predictor.predict(box=ebox, multimask_output=True)
        best_mask, info = select_best_mask(masks, scores, ebox)

        # 失败判定 -> 回退（box+点）
        need_fb = (info['cov'] < FAIL_COV_THRESH) or (info['area_norm'] < FAIL_AREA_NORM_THRESH)
        if need_fb:
            cx = 0.5 * (ebox[0] + ebox[2])
            cy = 0.5 * (ebox[1] + ebox[3])
            pos = np.array([[cx, cy]], dtype=np.float32)
            neg = sample_negative_points(ebox, best_mask, k=3)
            pts = np.concatenate([pos, neg], axis=0)
            labels = np.array([1] + [0] * len(neg), dtype=np.int32)

            masks2, scores2, _ = predictor.predict(
                box=ebox,
                point_coords=pts,
                point_labels=labels,
                multimask_output=True,
            )
            best_mask2, info2 = select_best_mask(masks2, scores2, ebox)
            # 选择更好的
            if info2['score'] > info['score']:
                best_mask, info = best_mask2, info2
            fallback_used += 1

        # 后处理
        best_mask, pp = postprocess_mask(best_mask, ebox)
        comps_total += pp['components_before']

        instances.append({
            'mask': best_mask,
            'box': ebox,
            'det_conf': float(det_conf),
            'cls': int(cls_id),
            'mask_score': float(info['pred_iou']),
            'cov': float(info['cov']),
            'area_norm': float(info['area_norm']),
            'edge_touch': float(info['edge_touch']),
            'score': float(info['score']),
            'pp_components_before': int(pp['components_before']),
        })

    kept, suppressed = mask_nms(instances)

    metrics = {
        'num_det': int(len(instances)),
        'num_kept': int(len(kept)),
        'suppressed': int(suppressed),
        'dup_rate': float(suppressed / max(1, len(instances))),
        'fallback_used': int(fallback_used),
        'fallback_rate': float(fallback_used / max(1, len(instances))),
        'avg_components_per_mask': float(comps_total / max(1, len(instances))),
    }
    return kept, metrics

def draw_instances(rgb: np.ndarray, instances: list[dict], names: dict | None = None, alpha: float = 0.55) -> np.ndarray:
    out = rgb.copy()
    rng = np.random.default_rng(123)
    for inst in instances:
        m = inst['mask']
        color = rng.integers(0, 255, size=(3,), dtype=np.uint8)
        out[m] = (out[m] * (1 - alpha) + color * alpha).astype(np.uint8)
        x1, y1, x2, y2 = inst['box'].astype(int)
        cv2.rectangle(out, (x1, y1), (x2, y2), (255, 0, 0), 2)
        label = str(inst['cls'])
        if names is not None and int(inst['cls']) in names:
            label = names[int(inst['cls'])]
        cv2.putText(out, f

## 你该怎么肉眼检查（建议）
- **SAM auto overlay**：看是否出现大量细碎小片/同一物体被切成很多块（mask 数量异常大）。
- **YOLO→SAM overlay**：看同一物体是否仍被多个实例重复抠（dup_rate 高），或经常只抠到局部（cov/area_norm 低）。
- 如果 fallback_rate 很高：优先检查 **YOLO 框是否过紧/漏检**，再调 `BOX_EXPAND_SCALE`。
- 如果 dup_rate 很高：检查 `YOLO_NMS_IOU` 与 `NMS_IOU_THRESH/CONTAIN_THRESH`（以及 YOLO 类别是否过宽泛）。